In [ ]:
 !pip install nltk scikit-learn pandas


In [ ]:
import nltk
import pandas as pd
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# Download required data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:

# ================================
# 1. SAMPLE DATA
# ================================
data = {
    "text": [
        "This is a sample sentence!",
        "Text cleaning is very important.",
        "NLP techniques are useful in real-world applications."
    ],
    "label": ["A", "B", "A"]
}

df = pd.DataFrame(data)
print("Original Data:\n", df)

Original Data:
                                                 text label
0                         This is a sample sentence!     A
1                   Text cleaning is very important.     B
2  NLP techniques are useful in real-world applic...     A


In [ ]:
# ================================
# 2. TEXT CLEANING
# ================================
def clean_text(text):
    text = text.lower()  # lowercase
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    tokens = word_tokenize(text)  # tokenize
    return tokens


df['tokens'] = df['text'].apply(clean_text)

In [ ]:
# ================================
# 3. REMOVE STOP WORDS
# ================================
stop_words = set(stopwords.words('english'))

df['no_stopwords'] = df['tokens'].apply(
    lambda words: [w for w in words if w not in stop_words]
)



In [ ]:
# ================================
# 4. LEMMATIZATION
# ================================
lemmatizer = WordNetLemmatizer()

df['lemmatized'] = df['no_stopwords'].apply(
    lambda words: [lemmatizer.lemmatize(w) for w in words]
)

# Convert back to sentence
df['clean_text'] = df['lemmatized'].apply(lambda words: " ".join(words))

In [ ]:
# ================================
# 5. LABEL ENCODING
# ================================
encoder = LabelEncoder()
df['encoded_label'] = encoder.fit_transform(df['label'])

In [ ]:
# ================================
# 6. TF-IDF REPRESENTATION
# ================================
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())



In [ ]:
# ================================
# 7. SAVE OUTPUTS
# ================================

# Save cleaned data
df.to_csv("cleaned_data.csv", index=False)

# Save TF-IDF matrix
tfidf_df.to_csv("tfidf_output.csv", index=False)

In [ ]:
# ================================
# 8. DISPLAY OUTPUTS
# ================================
print("\nCleaned Data:\n", df[['text', 'clean_text', 'encoded_label']])
print("\nTF-IDF Matrix:\n", tfidf_df)


Cleaned Data:
                                                 text  \
0                         This is a sample sentence!   
1                   Text cleaning is very important.   
2  NLP techniques are useful in real-world applic...   

                                   clean_text  encoded_label  
0                             sample sentence              0  
1                     text cleaning important              1  
2  nlp technique useful realworld application              0  

TF-IDF Matrix:
    application  cleaning  important       nlp  realworld    sample  sentence  \
0     0.000000   0.00000    0.00000  0.000000   0.000000  0.707107  0.707107   
1     0.000000   0.57735    0.57735  0.000000   0.000000  0.000000  0.000000   
2     0.447214   0.00000    0.00000  0.447214   0.447214  0.000000  0.000000   

   technique     text    useful  
0   0.000000  0.00000  0.000000  
1   0.000000  0.57735  0.000000  
2   0.447214  0.00000  0.447214  
